# Cointegration

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

In [2]:
close = pd.read_csv("../data/raw/close.csv", parse_dates=["Date"], index_col="Date")
log_price = np.log(close)

In [3]:
# candidate pairs, as defined in README.md
pairs = [
    ("XOM", "CVX"),
    ("XLE", "XOP"),
    ("NEM", "GOLD"),
    ("ADM", "BG"),
]

In [4]:
# hedge ratio via OLS on log prices: p_A = alpha + beta * p_B + spread
def compute_spread(a, b):
    y = log_price[a]
    x = sm.add_constant(log_price[b])
    model = sm.OLS(y, x).fit()
    alpha, beta = model.params["const"], model.params[b]
    spread = y - (alpha + beta * log_price[b])
    return spread, alpha, beta


spreads = {}
for a, b in pairs:
    spread, alpha, beta = compute_spread(a, b)
    spreads[(a, b)] = spread

spreads[("XOM", "CVX")].head()

Date
2015-01-02    0.194616
2015-01-05    0.207885
2015-01-06    0.203020
2015-01-07    0.213939
2015-01-08    0.207700
dtype: float64

In [5]:
# Engle-Granger cointegration test: regress p_A on p_B, then ADF-test the residual
# (statsmodels' coint() runs this two-step test with the correct EG critical values,
# rather than the plain ADF critical values, which don't apply to an estimated residual)
rows = []
for a, b in pairs:
    _, alpha, beta = compute_spread(a, b)
    t_stat, p_value, crit_values = coint(log_price[a], log_price[b])
    rows.append({
        "pair": f"{a} vs {b}",
        "beta": beta,
        "EG t-stat": t_stat,
        "p-value": p_value,
        "1% crit": crit_values[0],
        "5% crit": crit_values[1],
        "10% crit": crit_values[2],
    })

coint_summary = pd.DataFrame(rows).set_index("pair")
coint_summary

,beta,EG t-stat,p-value,1% crit,5% crit,10% crit
pair,,,,,,
XOM vs CVX,1.005379,-1.567709,0.734325,-3.900236,-3.338247,-3.045919
XLE vs XOP,0.778693,-0.703247,0.946287,-3.900236,-3.338247,-3.045919
NEM vs GOLD,0.387221,-1.689550,0.681661,-3.900236,-3.338247,-3.045919
ADM vs BG,0.884826,-2.460213,0.296892,-3.900236,-3.338247,-3.045919
